# 01 — Exploratory Data Analysis

This notebook explores the Bicing Barcelona 2024 dataset before modelling.
The goal is to understand occupancy patterns across stations, zones, seasons,
times of day, weather conditions and day types.

Outputs: plots saved to `reports/figures/`.

In [ ]:
import sys
sys.path.append('..')  # make src/ importable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils.config import PATHS

print('Libraries loaded OK')

---
## 1. Load processed data

In [ ]:
# Run 02_feature_engineering.ipynb first to generate this file
df = pd.read_parquet(DATASET_FILE)
df['datetime_hour'] = pd.to_datetime(df['datetime_hour'])

print(f'Rows:     {len(df):,}')
print(f'Stations: {df["station_id"].nunique()}')
print(f'Range:    {df["datetime_hour"].min()} → {df["datetime_hour"].max()}')

---
## 2. Occupancy distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Use horizon=6 rows only to avoid duplication in the distribution
df6 = df[df['horizon_hours'] == 6]

axes[0].hist(df6['current_occ'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of current occupancy')
axes[0].set_xlabel('Occupancy')

axes[1].hist(df6['target_occ'], bins=50, color='tomato', edgecolor='white')
axes[1].set_title('Distribution of target occupancy (T+6h)')
axes[1].set_xlabel('Occupancy')

plt.tight_layout()
plt.savefig(PATHS['figures'] / '01_occupancy_distribution.png', dpi=150)
plt.show()

---
## 3. Occupancy by hour and day type

In [ ]:
df6['day_type'] = 'Weekday'
df6.loc[df6['is_holiday'] == 1, 'day_type'] = 'Holiday'
df6.loc[(df6['is_weekend'] == 1) & (df6['is_holiday'] == 0), 'day_type'] = 'Weekend'

fig, ax = plt.subplots(figsize=(11, 5))
colors = {'Weekday': 'steelblue', 'Weekend': 'darkorange', 'Holiday': 'tomato'}
for dtype, color in colors.items():
    sub = df6[df6['day_type'] == dtype]
    hourly = sub.groupby('hour')['current_occ'].mean()
    ax.plot(hourly.index, hourly.values, label=dtype, color=color)

ax.set_xlabel('Hour of day')
ax.set_ylabel('Mean occupancy')
ax.set_title('Mean occupancy by hour and day type')
ax.legend()
plt.tight_layout()
plt.savefig(PATHS['figures'] / '02_hourly_by_daytype.png', dpi=150)
plt.show()

---
## 4. Heatmap: month × hour

In [ ]:
heatmap = df6.groupby(['month', 'hour'])['current_occ'].mean().unstack()

fig, ax = plt.subplots(figsize=(13, 6))
im = ax.imshow(heatmap.T, aspect='auto', cmap='RdYlGn_r', vmin=0.2, vmax=0.5)
plt.colorbar(im, label='Mean occupancy')
ax.set_title('Mean occupancy by month and hour — Barcelona Bicing 2024')
ax.set_xlabel('Month')
ax.set_ylabel('Hour')
ax.set_xticks(range(12))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.savefig(PATHS['figures'] / '03_heatmap_month_hour.png', dpi=150)
plt.show()

---
## 5. Occupancy vs weather

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, xlabel in zip(axes,
                            ['temperature', 'precipitation', 'windspeed'],
                            ['Temperature (°C)', 'Precipitation (mm)', 'Wind speed (km/h)']):
    sample = df6.sample(20_000, random_state=42)
    ax.scatter(sample[col], sample['current_occ'],
               alpha=0.05, s=3, color='steelblue')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Occupancy')

plt.suptitle('Occupancy vs weather features', y=1.02)
plt.tight_layout()
plt.savefig(PATHS['figures'] / '04_occupancy_vs_weather.png', dpi=150)
plt.show()

---
## 6. Station map (mean occupancy)

In [ ]:
station_occ = df6.groupby('station_id').agg(
    occ_mean=('current_occ', 'mean'),
    lat=('lat', 'first'),
    lon=('lon', 'first'),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 9))
sc = ax.scatter(station_occ['lon'], station_occ['lat'],
                c=station_occ['occ_mean'], cmap='RdYlGn_r',
                vmin=0.2, vmax=0.6, s=40, alpha=0.9)
plt.colorbar(sc, label='Mean annual occupancy')
ax.set_title('Mean annual occupancy per station — Barcelona 2024')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(PATHS['figures'] / '05_station_map.png', dpi=150)
plt.show()

---
## 7. Correlation matrix of model features

In [ ]:
from src.utils.config import FEATURES, TARGET

corr_cols = [c for c in FEATURES if c in df6.columns] + [TARGET]
corr = df6[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=90, fontsize=8)
ax.set_yticklabels(corr_cols, fontsize=8)
ax.set_title('Correlation matrix — model features')
plt.tight_layout()
plt.savefig(PATHS['figures'] / '06_correlation_matrix.png', dpi=150)
plt.show()